<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-06-function-calling/lesson-6.1-function-calling/practice/GCP_Capstone_6.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 6.1 — Gemini Function Calling

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Install the unified `google-genai` SDK, authenticate with Application Default Credentials (Colab), and create a Vertex client. Run this cell first — every exercise below depends on `client` and `types`.

In [ ]:
%%bash
pip install -q google-genai

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE to your project id

from google import genai
from google.genai import types

client = genai.Client(enterprise=True, project=PROJECT_ID,
                      location='global')
print(f'Client ready for {PROJECT_ID}')

### DocuMind tool functions

These three Python functions are the tools used throughout the lab. Each has type hints and a docstring — that is exactly what the SDK reads to build the schema Gemini sees. Run once.

In [ ]:
# DocuMind tool functions
def search_documents(query: str, doc_type: str = 'all', top_k: int = 5) -> dict:
    """Search DocuMind document collection by query.

    Args:
        query: Search query in natural language
        doc_type: Filter by document type (research_paper, invoice, legal, form, all)
        top_k: Number of results to return
    """
    # Mock: in production, call VECTOR_SEARCH
    return {'results': [
        {'doc_id': 'D-42', 'title': 'Refund Policy v3', 'relevance': 0.94},
        {'doc_id': 'D-17', 'title': 'Return Guidelines', 'relevance': 0.87}
    ], 'total': 2}

def calculate_processing_cost(page_count: int, file_type: str = 'pdf') -> dict:
    """Estimate document processing cost in USD and INR.

    Args:
        page_count: Number of pages in the document
        file_type: File format (pdf, html, text, docx)
    """
    rates = {'pdf': 0.07, 'html': 0.03, 'text': 0.02, 'docx': 0.05}
    cost = page_count * rates.get(file_type, 0.07)
    return {'cost_usd': round(cost, 2), 'cost_inr': round(cost * 85, 2),
            'breakdown': f'{page_count} pages x ${rates.get(file_type, 0.07)}/page'}

def get_usage_stats(metric: str, days: int = 7) -> dict:
    """Get DocuMind RAG pipeline usage statistics.

    Args:
        metric: Which metric to retrieve (queries, costs, latency, users)
        days: Number of days to look back
    """
    mock_data = {'queries': 1247, 'costs': 18.50, 'latency': 245, 'users': 42}
    return {'metric': metric, 'period': f'last {days} days',
            'value': mock_data.get(metric, 0), 'trend': '+12%'}

TOOLS = [search_documents, calculate_processing_cost, get_usage_stats]
print(f'Defined {len(TOOLS)} DocuMind tools')

## Exercise 1: First FunctionDeclaration

**Difficulty:** Easy

Define `search_documents`. Send to Gemini. Verify `function_call` in response.

1. Create FunctionDeclaration with name, description, parameters
2. Wrap in `types.Tool`
3. Send query, check `response.function_calls`

In [ ]:
from google.genai.types import FunctionDeclaration, Tool

# Step 1: hand-write the FunctionDeclaration (name, description, JSON-schema parameters)
search_decl = FunctionDeclaration(
    name='search_documents',
    description='Search DocuMind documents by query.',
    parameters={
        'type': 'object',
        'properties': {
            'query': {'type': 'string', 'description': 'Search query'},
            'doc_type': {'type': 'string', 'enum': ['research_paper', 'invoice', 'legal', 'form', 'all']}
        },
        'required': ['query']
    }
)

# Step 2: wrap it in a Tool
tool = Tool(function_declarations=[search_decl])

# Step 3: send a query and inspect the function call the model produced
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Find our refund policy documents',
    config=types.GenerateContentConfig(tools=[tool])
)

fc = response.function_calls[0]
print(f'name: {fc.name}')
print(f'args: {fc.args}')
assert fc.name == 'search_documents'
print('OK: model requested search_documents')

## Exercise 2: Manual Execution Loop

**Difficulty:** Easy

Complete the 4-step loop: send → function_call → execute → return result.

1. Check `response.function_calls`
2. Execute Python function with `**fc.args`
3. Create FunctionResponse, send back
4. Print final text answer

In [ ]:
# Reuse the `tool` from Exercise 1. Full manual 4-step loop.
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Find our refund policy documents',
    config=types.GenerateContentConfig(tools=[tool])
)

print('=== Manual Loop ===')
if response.function_calls:
    fc = response.function_calls[0]           # Step 1: model asked for a call
    print(f'Function: {fc.name}')
    print(f'Args: {fc.args}')

    result = search_documents(**fc.args)       # Step 2: you execute it
    print(f'Result: {result}')

    # Steps 3 + 4: send the result back as a FunctionResponse, get final NL answer
    final = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=[
            types.Content(role='user', parts=[
                types.Part.from_text(text='Find our refund policy documents')]),
            response.candidates[0].content,
            types.Content(role='user', parts=[
                types.Part.from_function_response(
                    name=fc.name, response=result)])
        ],
        config=types.GenerateContentConfig(tools=[tool])
    )
    print(f'\nFinal answer: {final.text}')

## Exercise 3: Automatic Function Calling

**Difficulty:** Easy

Pass Python functions directly to `tools=[]`. Verify auto-execution.

1. Define functions with type hints + docstrings
2. Pass to `tools=[]` directly
3. Check `response.text` for final answer

In [ ]:
# Pass the raw Python callables (TOOLS) — the SDK builds the schema, calls the
# function, feeds the result back, and returns the final text in one shot.
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='How much would it cost to process a 45-page PDF?',
    config=types.GenerateContentConfig(tools=TOOLS)
)
print('=== Auto Function Calling ===')
print(response.text)

## Exercise 4: 3-Tool Selection

**Difficulty:** Medium

Define 3 tools. Test 3 queries. Verify the correct tool is selected each time.

1. `search_documents`, `calculate_processing_cost`, `get_usage_stats`
2. Send queries targeting each tool
3. Verify `function_calls[0].name` matches expected

In [ ]:
# Same TOOLS list, three queries that should each route to a different tool
# (plus a greeting that should route to NONE of them).
queries = [
    'What does our refund policy say?',
    'How much to process 200 pages of invoices?',
    'How many queries did we get last week?',
    'Hello, how are you today?',  # Should NOT call a function
]

for q in queries:
    r = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=q,
        config=types.GenerateContentConfig(tools=TOOLS)
    )
    if r.function_calls:
        print(f'Q: {q}')
        print(f'  Tool: {r.function_calls[0].name}({r.function_calls[0].args})')
    else:
        print(f'Q: {q}')
        print(f'  Text: {r.text[:80]}...')
    print()

## Exercise 5: AUTO vs ANY vs NONE

**Difficulty:** Medium

Test all 3 modes. Greeting in AUTO (text). Greeting in ANY (forced call). NONE (text only).

1. Set mode in `tool_config.function_calling_config`
2. Send same greeting in each mode
3. Compare: AUTO=text, ANY=function_call, NONE=text

In [ ]:
def call_with_mode(mode):
    return client.models.generate_content(
        model='gemini-3.6-flash',
        contents='Hello!',
        config=types.GenerateContentConfig(
            tools=TOOLS,
            tool_config=types.ToolConfig(
                function_calling_config=types.FunctionCallingConfig(mode=mode)))
    )

# AUTO: model decides -> a greeting needs no tool, so it answers in text
r_auto = call_with_mode('AUTO')
print(f'AUTO  -> calls={len(r_auto.function_calls or [])}  text={r_auto.text[:60]!r}')

# ANY: model is FORCED to call some tool, even for a greeting
r_any = call_with_mode('ANY')
if r_any.function_calls:
    fc = r_any.function_calls[0]
    print(f'ANY   -> forced call: {fc.name}({fc.args})')

# NONE: tools are disabled -> always plain text, never a call
r_none = call_with_mode('NONE')
print(f'NONE  -> calls={len(r_none.function_calls or [])}  text={r_none.text[:60]!r}')

## Exercise 6: Parallel Function Calls

**Difficulty:** Medium

Trigger 2 independent calls. Execute both. Return both results.

1. Send a query requiring 2 tools
2. Check `len(response.function_calls) == 2`
3. Execute both, send both FunctionResponse objects

In [ ]:
# One prompt that needs two independent tools: a cost estimate AND usage stats.
# Manual loop so we can see both calls arrive in a single response.
REGISTRY = {
    'search_documents': search_documents,
    'calculate_processing_cost': calculate_processing_cost,
    'get_usage_stats': get_usage_stats,
}

prompt = 'What did it cost to process a 45-page PDF, and how many queries did we get last week?'

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=prompt,
    config=types.GenerateContentConfig(
        tools=TOOLS,
        # Disable auto-exec: we run the manual parallel loop ourselves, so the
        # forced calls must come back UNEXECUTED instead of the SDK running them.
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        # Force tool use so both calls come back together instead of one-at-a-time
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(mode='ANY')))
)

calls = response.function_calls or []
print(f'Parallel calls returned: {len(calls)}')

# Execute every call, collect one FunctionResponse part per call
response_parts = []
for fc in calls:
    result = REGISTRY[fc.name](**fc.args)
    print(f'  {fc.name}({fc.args}) -> {result}')
    response_parts.append(
        types.Part.from_function_response(name=fc.name, response=result))

# Send all results back in a single turn so the model can synthesise one answer
final = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=[
        types.Content(role='user', parts=[types.Part.from_text(text=prompt)]),
        response.candidates[0].content,
        types.Content(role='user', parts=response_parts),
    ],
    config=types.GenerateContentConfig(tools=TOOLS)
)
print(f'\nSynthesised answer: {final.text}')

## Exercise 7: Chat with Tools

**Difficulty:** Challenge

Build a multi-turn chat. Test 5 queries with context across turns.

1. `client.chats.create()` with tools and `system_instruction`
2. Send 5 sequential messages
3. Verify context carries across turns

In [ ]:
# A chat session keeps history for you; tools auto-execute each turn.
chat = client.chats.create(
    model='gemini-3.6-flash',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        system_instruction='You are DocuMind AI. Use tools to answer questions.'
    )
)

turns = [
    'Find our refund policy',
    'How much would it cost to process that document if it is 12 pages?',
    'What are our query stats for this week?',
    'And what about latency over the same period?',
    'Summarise everything you found for me.',  # relies on earlier context
]

for i, msg in enumerate(turns, start=1):
    r = chat.send_message(msg)
    print(f'Turn {i}: {r.text[:120]}...')
    print()

print(f'Chat history: {len(chat.get_history())} messages')

## Exercise 8: Error-Safe Dispatcher

**Difficulty:** Challenge

Build `safe_execute` with validation, error handling, and destructive-op blocking.

1. Registry of allowed functions
2. Block destructive operations
3. `try`/`except` with error dict return

In [ ]:
# Production-safe function dispatcher: never blindly call whatever the model names.
def safe_execute(fc):
    ALLOWED = {
        'search_documents': search_documents,
        'calculate_processing_cost': calculate_processing_cost,
        'get_usage_stats': get_usage_stats,
    }
    DESTRUCTIVE = {'delete_document', 'send_email', 'modify_access'}

    if fc.name in DESTRUCTIVE:
        return {'error': f'Operation {fc.name} requires manual confirmation'}
    if fc.name not in ALLOWED:
        return {'error': f'Unknown function: {fc.name}'}
    try:
        return ALLOWED[fc.name](**fc.args)
    except Exception as e:
        return {'error': f'Execution failed: {str(e)}'}

# Test safe execution with a good call, an unknown func, and a bad-arg call
print('Safe execute tests:')
for name, args in [('search_documents', {'query': 'test'}),
                   ('unknown_func', {}),
                   ('calculate_processing_cost', {'page_count': 'abc'})]:
    class MockFC:
        pass
    fc = MockFC()
    fc.name = name
    fc.args = args
    result = safe_execute(fc)
    print(f'  {name}: {result}')